# Mesh Creation, Smoothing, and Refinement

Note: you will need to install the brain_mesh_creation module to run this notebook. 

### Setup paths

In [2]:
from pathlib import Path
import os
import sys

# Project root: assumes notebook is run from notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent

# Main folders
SRC_DIR = PROJECT_ROOT / "src"
PACKAGE_DIR = SRC_DIR / "brain_mesh_creation"
wrk_dir = str(SRC_DIR / "dependencies")
sub_dir = str(PROJECT_ROOT / "data" / "subjects")

# Subject and output names
subject_folder = "sub0045"
output_name = "output"

# Make sure src/ is importable
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from brain_mesh_creation import bmctk
from brain_mesh_creation import mesh_utils

print("PROJECT_ROOT:", PROJECT_ROOT)
print("subject_folder:", subject_folder)

PROJECT_ROOT: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation
wrk_dir: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/src/dependencies
sub_dir: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects
subject_folder: sub0045


### Meshing parameters setup

We recommend `mesh_size = 2` for the example subject. Coarser settings such as `3` may produce meshes that are too irregular for the smoothing executable and can cause a Bus error during smoothing. High-resolution settings such as `1` or `1.5` can cause the resulting mesh file too large - but feel free to test other mesh sizes. 

In [15]:
# Set mesh size (in mm)
mesh_size = 2

# Voxel correction variables
# It is generally recommended not to change these unless needed
iterations = 3
blocks = 1
threshold = 0.9  # Threshold for neighbour disagreement during voxel correction

# Manual centre-of-gravity edits
# Used for generating the centre of gravity for TBI simulations
cog_id = 80000000
change_ap = 0
change_lr = 0
change_is = 0
change_cog = [change_ap, change_lr, change_is]

## Mesh creation

The segmented image is converted into mesh using a direct voxel-to-element approach, generating a fully hexahedral mesh. Hexahedral elements are selected over tetrahedra due to their superior performance in explicit simulations, specifically regarding time-step stability and resistance to volumetric locking in nearly incompressible materials such as brain tissue (Tadepalli et al. 2011; Giudice et al. 2019).

To accommodate varying computational resources and fidelity requirements, the pipeline incorporates a customizable resampling module. While the default settings maintains native MRI resolution, the pipeline allows users to define a custom target element size (e.g., 1mm, 1.5 mm or 2.0 mm) prior to mesh generation. The segmented masks are automatically resampled to this user-defined voxel size, allowing users to optimize the balance between geometric accuracy and computational cost based on specific study requirements.

### Algorithmic reconstruction of meningeal structures
Following the solid mesh generation, the meningeal layers are constructed. Standard MRI-based FE meshes frequently omit meningeal structures due to their thin geometry, which is often indistinguishable at standard MRI resolutions (Chen and Ostoja-Starzewski 2010). However, these structures have shown to play a significant role in brain biomechanics by limiting the relative motion of left and right hemispheres or constraining the upward movements of the corpus callosum (Hernandez et al. 2019; Darvishi et al. 2025). Although some studies have attempted to incorporate these structures manually in the brain model, these approaches compromise the automation which is vital for large-scale patient-specific modeling (Miller et al. 2016). To address this, our pipeline algorithmically synthesizes them based on anatomical landmarks:

- Falx: The algorithm identifies the longitudinal fissure separating left and right hemispheres. A shell layer is generated in the mid-sagittal plane, extending from the corpus callosum to the interior surface of skull.

- Tentorium: The interface between the inferior surface of the occipital/temporal lobes and the superior surface of the cerebellum is identified to generate a partition representing the tentorium.

- Pia and Dura mater: The pia mater is generated as a shell layer coincident with the outer surface of the brain parenchyma, while the dura mater is generated from the inner surface of the skull mesh.

In [18]:
# Only run this cell if you have previously generated a mesh AND you want to discard it

%%bash -s "$sub_dir" "$subject_folder" "$output_name"

echo "looking at" "$2"
cd "$1"/"$2"
echo "Removing legacy mesh data for $2"
rm -r "$3"

looking at sub0045
Removing legacy mesh data for sub0045


In [19]:
import os

subject_path = os.path.join(sub_dir, subject_folder)
pre_model_path = os.path.join(subject_path, "pre_model.nii.gz")

print("Looking at:", subject_path)

if os.path.isdir(subject_path):

    if not os.path.isfile(pre_model_path):
        print(f"Cannot find pre_model.nii.gz at: {pre_model_path}")
        print("Please run notebook 01 first to generate pre_model.nii.gz")
    else:
        print("Starting mesh generation for", subject_folder)

        # Move into subject folder so output files are written there
        os.chdir(subject_path)
        print("Working directory:", os.getcwd())

        # Initialise model creation
        my_model = bmctk.create_model("pre_model.nii.gz", output_name, wrk_dir)

        # Create relative coordinates
        my_model.create_coordinates()

        # Resample model if necessary
        my_model.resample(mesh_size)

        # Assign missing grey matter
        my_model.gm_check()

        # Check contact of grey matter to skull
        my_model.check_contact(257, 256, 42)
        my_model.check_contact(257, 256, 3)
        my_model.check_contact(257, 256, 47)
        my_model.check_contact(257, 256, 8)

        # Check contact of CSF to void
        my_model.check_contact(0, 257, 256)

        # Check contact of CSF to white matter
        my_model.check_contact(41, 42, 256)
        my_model.check_contact(2, 3, 256)
        my_model.check_contact(46, 47, 256)
        my_model.check_contact(7, 8, 256)

        # Make corrections for voxels inside other parts
        my_model.voxel_corrections(iterations, threshold, blocks)

        # Create falx
        LR_center_coords = my_model.create_falx()
        LR_center = LR_center_coords[0]

        # Create tentorium
        my_model.create_tentorium()

        # Create mesh
        max_node = my_model.create_k_mesh()
        print("Largest node number assigned:", max_node)

        # Create set list
        my_model.set_list()

        print("Finished making model for", subject_folder)
        print()
else:
    print("Cannot see folder", subject_folder, "in", sub_dir)

Looking at: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045
Starting mesh generation for sub0045
Working directory: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045
Reading brain data... 
Assigning missing brain matter to grey matter and brain stem... 
Number of missing voxels: 1033
Number of missing voxels: 887
Number of missing voxels: 847
Number of missing voxels: 833
Number of missing voxels: 827
Number of missing voxels: 822
Number of missing voxels: 822
Assigning remaining voxels... 
Number of missing voxels: 0
Voxel correction: 
Iteration 1
Values changed: 112

Voxel correction: 
Iteration 2
Values changed: 2

Voxel correction: 
Iteration 3
Values changed: 0

Falx location voxel: 45.51285702016037
Falx location scanner: 0.025714040320735876
Reading nodal coordinates... 
Assigning element nodes... 
Writting K file...
Largest node number assigned: 296494

## Mesh smoothing
A known limitation of direct voxel-to-element conversion is the creation of stair-step surface artifacts. These interfaces, particularly at the outer boundaries of the brain, can lead to artificial stress concentrations and numerical instabilities. To mitigate this, we applied a Laplacian smoothing algorithm adapted from the methodology of Chen and Ostoja-Starzewski (Chen and Ostoja-Starzewski 2010). The algorithm operates on the boundary surface nodes of the generated mesh through an iterative process. This process effectively removes the stair-steps shape on the boundaries, while constraining the total volume change to minimal levels. This step is vital for ensuring that the interfaces between the brain regions, meninges, CSF and skull are smooth, thereby preventing unrealistic stress concentration during simulation.

In [20]:
%%bash -s "$mesh_size" "$wrk_dir" "$sub_dir" "$subject_folder" "$output_name"

set -e

MESH_SIZE="$1"
WRK_DIR="$2"
SUB_DIR="$3"
SUBJECT="$4"
OUTPUT_NAME="$5"

SUBJECT_PATH="${SUB_DIR}/${SUBJECT}"
OUTPUT_PATH="${SUBJECT_PATH}/${OUTPUT_NAME}"

# Choose the mesh smoother executable that fits your system. 
# If you are using Codespace, choose the Linux version. 
# mesh_smoother_linux_x86_64" # Linux executable 
# mesh_smoother_macos_x86_64 # MacOS Intel executable
# mesh_smoother_windows_x86_64.exe # Windows executable
SMOOTHER_PATH="${WRK_DIR}/rs/mesh_smoother_macos_arm64" # MacOS arm64 executable

echo "Looking at ${SUBJECT}"
cd "${SUBJECT_PATH}" || exit 1
pwd
echo "Moved to ${SUBJECT}"

if [ ! -x "${AOUT_PATH}" ]; then
    echo "Cannot find smoothing executable or it is not executable: ${SMOOTHER_PATH}"
    exit 1
fi

if [ ! -f "${OUTPUT_PATH}/mesh.k" ]; then
    echo "Cannot find input mesh: ${OUTPUT_PATH}/mesh.k"
    exit 1
fi

cd "${OUTPUT_PATH}" || exit 1
echo "Running mesh smoothing in $(pwd)"

"${SMOOTHER_PATH}" mesh.k mesh_smoothed.k 8 0.2 200 2000 "${MESH_SIZE}"

echo "Created: ${OUTPUT_PATH}/mesh_smoothed.k"
cd ..

Looking at sub0045
/Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045
Moved to sub0045
Running mesh smoothing in /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045/output
Dilatational wave speed [m/s] in: 
skin * cortical bone * diploe * CSF * brain * Falx * Tentorium * pia matter *
206.475 * 2873.84 * 1755.82 * 154.94 * 220.481 * 188.383 * 0 * 
Reading the input file ...

Number of solid elements: 370618
Number of shell elements: 87485
Number of nodes: 389919
Largest node number: 2964944

Reading the input file finished.

Node classification and neighbour listing ...

End of classification and neighbour listing.

Applying the smoothing filter ...

1 iterations done.
Minimum Jacobian (at element centre): 0.5904
Minimum characteristic length (2D - 3D): 1.48568 - 1.3608
Minimum time step: 0.000521809
2 iterations done.
Minimum Jacobian (at element centre): 0.353085


## Mesh refinement: Brain-Skull Contact Prevention

Direct voxel-to-element conversion and subsequent mesh smoothing can result in brain surface nodes becoming shared with the skull mesh, creating sites of non-physical brain-skull contact where no CSF layer is present. As illustrated in Fig. 3, these contact artefacts produce unrealistic concentrations of maximum principal Green-Lagrange strain in the brain parenchyma under loading, which would compromise the physical validity of simulation results. To resolve this, the pipeline includes a contact repair module that iterates through the brain surface boundary and detects any brain nodes shared with the skull mesh. All skull elements associated with these shared nodes are flagged and reclassified as CSF, restoring a continuous CSF layer at the brain-skull interface and ensuring a strict physical separation between the brain parenchyma and the skull. 

In [21]:
import os
import sys
from pathlib import Path

# Make sure src/ is importable
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from brain_mesh_creation import mesh_utils

current_subject_dir = os.path.join(sub_dir, subject_folder)
mesh_output_dir = os.path.join(current_subject_dir, output_name)

input_k_file = os.path.join(mesh_output_dir, "mesh_smoothed.k")
output_k_file = os.path.join(mesh_output_dir, "mesh_smoothed_revised.k")

print(f"--- Processing Subject: {subject_folder} ---")
print("Applying brain-skull interface regularization...")

if not os.path.isfile(input_k_file):
    print(f"Cannot find input mesh file: {input_k_file}")
else:
    mesh_utils.create_csf_buffer(input_k_file, output_k_file)
    print("Finished creating revised mesh.")
    # print(output_k_file)

--- Processing Subject: sub0045 ---
Applying brain-skull interface regularization...
Input: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045/output/mesh_smoothed.k
Output: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045/output/mesh_smoothed_revised.k
Processing mesh: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045/output/mesh_smoothed.k
   -> Identifying Brain Nodes...
   -> Checking for Brain-Skull/Skin contact...
Writing result to: /Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045/output/mesh_smoothed_revised.k
Success. Total elements converted to CSF buffer: 25595
Finished creating revised mesh:
/Users/yc4421/Library/CloudStorage/Box-Box/GTA/ReCoDE_BrainMesh/ReCoDE-brain-mesh-creation/data/subjects/sub0045/output/mesh_smoothed_rev

## References
- Tadepalli SC, Erdemir A, Cavanagh PR (2011) Comparison of hexahedral and tetrahedral elements in finite element analysis of the foot and footwear. J Biomech 44:2337–2343
- Giudice JS, Zeng W, Wu T, et al (2019) An analytical review of the numerical methods used for finite element modeling of traumatic brain injury. Ann Biomed Eng 47:1855–1872
- Chen Y, Ostoja-Starzewski M (2010) MRI-based finite element modeling of head trauma: spherically focusing shear waves. Acta Mech 213:155–167 
- Betsholtz C, Engelhardt B, Koh GY, et al (2024) Advances and controversies in meningeal biology. Nat Neurosci 27:2056–2072
- Hernandez F, Giordano C, Goubran M, et al (2019) Lateral impacts correlate with falx cerebri displacement and corpus callosum trauma in sports-related concussions. Biomech Model Mechanobiol 18:631–649
- Darvishi V, Del Giovane M, David MCB, et al (2025) Towards a Deeper Understanding of Axonal Injury Mechanisms Using a Novel Biomechanical Model of Normal Pressure Hydrocephalus. 20th Injury Biomechanics Symposium
- Miller LE, Urban JE, Stitzel JD (2016) Development and validation of an atlas-based finite element brain model. Biomech Model Mechanobiol 15:1201–1214
- 